# Phase 4: Feature Engineering
Uses src.preprocess and src.feature_builder to create a processed dataset.

In [ ]:
import pandas as pd
import sys
import os
import pickle
sys.path.append(os.path.abspath('..'))

from src.preprocess import Preprocessor
from src.feature_builder import build_features

# Read data
train_transaction = pd.read_csv('../data/raw/train_transaction.csv', nrows=100000)
train_identity = pd.read_csv('../data/raw/train_identity.csv')
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

# Feature engineering first (can add new columns)
train = build_features(train)

# Separate target
y = train['isFraud']
X = train.drop(columns=['isFraud', 'TransactionID', 'TransactionDT'], errors='ignore')

# Preprocess (Drop >85% nulls, impute, encode)
preprocessor = Preprocessor()
X_processed = preprocessor.fit(X).transform(X)

# Save artifacts
with open('../model/encoders.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

feature_columns = X_processed.columns.tolist()
with open('../model/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

# Reattach target for processed save
X_processed['isFraud'] = y
X_processed.to_csv('../data/processed/train_sample_processed.csv', index=False)
print('Feature engineering complete. Shape:', X_processed.shape)